# Dataset explorer

Browse the queries and gold chunks written by `prepare_data.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("/home/htkumar/agentic_rl/search_agent/data")

queries = pd.read_parquet(DATA_DIR / "queries.parquet")
queries["facts"] = queries["facts"].map(json.loads)
queries["num_facts"] = queries["facts"].map(len)
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet").set_index("chunk_id")["text"]

print(f"corpus: {len(corpus):,} chunks")
queries.groupby(["split", "num_facts"]).size().unstack()

In [ ]:
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 100)
queries[["query_id", "split", "num_facts", "query"]].head()

In [ ]:
def show(query_id: str) -> None:
    """Print a query, its answer, and each fact with the text of its gold chunks."""
    row = queries.set_index("query_id").loc[query_id]
    print(f"{query_id} ({row['split']}, {row['num_facts']} facts)\n")
    print(row["query"], "\n")
    print("Answer:", row["answer"], "\n")
    for i, fact in enumerate(row["facts"], 1):
        final = " [final answer]" if fact["is_final_answer"] else ""
        print(f"=== Fact {i}{final}: {fact['fact']}")
        for chunk_id in fact["chunk_ids"]:
            print(f"\n[{chunk_id}]\n{corpus[chunk_id]}\n")


show(queries["query_id"].iloc[0])